# Testing Conversation Agent with Note Tools

This notebook demonstrates the conversation agent using the note-taking system.

## Features Tested:
- Creating notes via natural language
- Semantic search with vector embeddings
- Updating and managing notes
- Multi-search capabilities
- Error handling with plain English feedback

## Setup

In [1]:
import asyncio
import os
from datetime import UTC, datetime

# Add project root to path
import sys
sys.path.insert(0, os.path.abspath("."))

from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Environment loaded")

✅ Environment loaded


## Initialize In-Memory Repository (for testing)

We'll use an in-memory repository for this demo to avoid database setup.

In [2]:
from typing import Any
from datetime import datetime, UTC
from src.domain.entities.note import Note

class InMemoryNoteRepository:
    """In-memory note repository for testing."""
    
    def __init__(self):
        self._notes: dict[str, Note] = {}
    
    async def create(self, note: Note) -> Note:
        if note.id in self._notes:
            raise ValueError(f"Note with ID {note.id} already exists")
        self._notes[note.id] = note
        return note
    
    async def get_by_id(self, user_id: str, note_id: str) -> Note | None:
        note = self._notes.get(note_id)
        if note and note.user_id == user_id:
            return note
        return None
    
    async def update(self, note: Note) -> Note:
        if note.id not in self._notes:
            raise ValueError(f"Note with ID {note.id} not found")
        self._notes[note.id] = note
        return note
    
    async def delete(self, user_id: str, note_id: str) -> bool:
        note = self._notes.get(note_id)
        if note and note.user_id == user_id:
            del self._notes[note_id]
            return True
        return False
    
    async def list_by_user(self, user_id: str, limit: int = 10, offset: int = 0) -> list[Note]:
        user_notes = [n for n in self._notes.values() if n.user_id == user_id]
        user_notes.sort(key=lambda n: n.updated_at, reverse=True)
        return user_notes[offset:offset + limit]
    
    async def search_by_vector(
        self, 
        user_id: str, 
        query_embedding: list[float],
        limit: int = 5,
        similarity_threshold: float = 0.7
    ) -> list[tuple[Note, float]]:
        """Simple vector search using cosine similarity."""
        import math
        
        def cosine_similarity(a: tuple[float, ...], b: list[float]) -> float:
            dot_product = sum(x * y for x, y in zip(a, b))
            magnitude_a = math.sqrt(sum(x * x for x in a))
            magnitude_b = math.sqrt(sum(x * x for x in b))
            if magnitude_a == 0 or magnitude_b == 0:
                return 0.0
            return dot_product / (magnitude_a * magnitude_b)
        
        results = []
        for note in self._notes.values():
            if note.user_id != user_id or note.embedding is None:
                continue
            
            similarity = cosine_similarity(note.embedding, query_embedding)
            if similarity >= similarity_threshold:
                results.append((note, similarity))
        
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:limit]
    
    async def search_by_keyword(self, user_id: str, keyword: str, limit: int = 10) -> list[Note]:
        keyword_lower = keyword.lower()
        results = []
        
        for note in self._notes.values():
            if note.user_id != user_id:
                continue
            if keyword_lower in note.title.lower() or keyword_lower in note.content.lower():
                results.append(note)
        
        results.sort(key=lambda n: n.updated_at, reverse=True)
        return results[:limit]
    
    async def search_by_tags(
        self, 
        user_id: str, 
        tags: list[str],
        match_all: bool = False,
        limit: int = 10
    ) -> list[Note]:
        results = []
        
        for note in self._notes.values():
            if note.user_id != user_id:
                continue
            
            note_tags = set(note.tags)
            search_tags = set(tags)
            
            if match_all:
                if search_tags.issubset(note_tags):
                    results.append(note)
            else:
                if note_tags.intersection(search_tags):
                    results.append(note)
        
        results.sort(key=lambda n: n.updated_at, reverse=True)
        return results[:limit]
    
    async def search_by_date_range(
        self,
        user_id: str,
        date_from: datetime | None = None,
        date_to: datetime | None = None,
        limit: int = 10
    ) -> list[Note]:
        results = []
        
        for note in self._notes.values():
            if note.user_id != user_id:
                continue
            
            if date_from and note.created_at < date_from:
                continue
            if date_to and note.created_at > date_to:
                continue
            
            results.append(note)
        
        results.sort(key=lambda n: n.created_at, reverse=True)
        return results[:limit]
    
    async def count_by_user(self, user_id: str) -> int:
        return sum(1 for n in self._notes.values() if n.user_id == user_id)

print("✅ In-memory repository created")

✅ In-memory repository created


## Initialize Embedding Service

In [3]:
from src.infrastructure.llm.embedding.gemini_embedding_service import GeminiEmbeddingService
from src.config.settings import settings

# Initialize Gemini embedding service
embedding_service = GeminiEmbeddingService(
    api_key=settings.gemini_api_key,
    model_name=settings.embedding_model
)

print(f"✅ Embedding service initialized: {embedding_service.get_model_name()}")
print(f"   Embedding dimension: {embedding_service.get_dimension()}")

✅ Embedding service initialized: gemini-embedding-001
   Embedding dimension: 1536


/Users/xavierau/Code/python/whatsapp_chatbot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configure Note Tools

In [4]:
from src.infrastructure.llm.tools.notes import configure_note_tools, set_user_context

# Create repository instance
repository = InMemoryNoteRepository()

# Configure note tools with dependencies
configure_note_tools(
    repository=repository,
    embedding_service=embedding_service
)

# Set user context
set_user_context("test-user-123")

print("✅ Note tools configured")
print("   Repository: In-Memory")
print("   User context: test-user-123")

✅ Note tools configured
   Repository: In-Memory
   User context: test-user-123


## Initialize Conversation Agent

In [5]:
from src.infrastructure.llm.conversation_agent import ConversationAgent
from src.application.interfaces.conversation_agent import ConversationContext

# Initialize agent
agent = ConversationAgent(
    model=settings.default_llm_model,
    temperature=0.7,
    max_tokens=2000
)

print("✅ Conversation agent initialized")
print(f"   Model: {agent.model}")
print(f"   Temperature: {agent.temperature}")
print(f"   Available tools: compute, jot, note_tools")

✅ Conversation agent initialized
   Model: gemini/gemini-2.5-flash
   Temperature: 0.7
   Available tools: compute, jot, note_tools


## Helper Function for Testing

In [6]:
def ask_agent(message: str, history: list = None) -> str:
    """Helper function to interact with the agent."""
    from src.application.interfaces.conversation_agent import Message
    
    # Convert history to Message objects
    message_history = tuple()
    if history:
        message_history = tuple(
            Message(role=msg["role"], content=msg["content"]) 
            for msg in history
        )
    
    # Create context
    context = ConversationContext(
        current_message=message,
        history=message_history
    )
    
    # Get response
    response = agent.generate_response(context)
    
    return response.content

print("✅ Helper function ready")

✅ Helper function ready


## Test 1: Creating Notes

In [7]:
print("\n" + "="*80)
print("TEST 1: Creating Notes")
print("="*80 + "\n")

# Test creating a note
user_message = "Please save a note: 'Q4 Planning Meeting - Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on improving user onboarding.' Tag it with 'work', 'meeting', 'q4'"

print(f"User: {user_message}")
print("\nAgent:")

response = ask_agent(user_message)
print(response)

print(f"\n📊 Notes in repository: {await repository.count_by_user('test-user-123')}")


TEST 1: Creating Notes

User: Please save a note: 'Q4 Planning Meeting - Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on improving user onboarding.' Tag it with 'work', 'meeting', 'q4'

Agent:
response_content I've saved your note titled 'Q4 Planning Meeting' with the content you provided and tagged it with 'work', 'meeting', and 'q4'.
I've saved your note titled 'Q4 Planning Meeting' with the content you provided and tagged it with 'work', 'meeting', and 'q4'.

📊 Notes in repository: 1


## Test 2: Creating Multiple Notes

In [8]:
print("\n" + "="*80)
print("TEST 2: Creating Multiple Notes")
print("="*80 + "\n")

notes_to_create = [
    "Save a note: 'Shopping List - milk, eggs, bread, coffee, bananas' with tags 'personal', 'shopping'",
    "Create a note titled 'Python Best Practices' with content 'Always use type hints, follow PEP 8, write docstrings, use virtual environments' and tags 'programming', 'python'",
    "Remember this: 'Doctor appointment on Friday at 2pm - bring insurance card' tags: 'personal', 'health'"
]

for i, message in enumerate(notes_to_create, 1):
    print(f"\n--- Note {i} ---")
    print(f"User: {message}")
    print("\nAgent:")
    response = ask_agent(message)
    print(response)

print(f"\n📊 Total notes in repository: {await repository.count_by_user('test-user-123')}")


TEST 2: Creating Multiple Notes


--- Note 1 ---
User: Save a note: 'Shopping List - milk, eggs, bread, coffee, bananas' with tags 'personal', 'shopping'

Agent:
response_content I've saved your note titled 'Shopping List' with the content 'Shopping List - milk, eggs, bread, coffee, bananas' and tags 'personal' and 'shopping'.
I've saved your note titled 'Shopping List' with the content 'Shopping List - milk, eggs, bread, coffee, bananas' and tags 'personal' and 'shopping'.

--- Note 2 ---
User: Create a note titled 'Python Best Practices' with content 'Always use type hints, follow PEP 8, write docstrings, use virtual environments' and tags 'programming', 'python'

Agent:
response_content I've created a note titled 'Python Best Practices' with the content you provided and the tags 'programming' and 'python'.
I've created a note titled 'Python Best Practices' with the content you provided and the tags 'programming' and 'python'.

--- Note 3 ---
User: Remember this: 'Doctor appointment

## Test 3: Listing Notes

In [9]:
print("\n" + "="*80)
print("TEST 3: Listing Notes")
print("="*80 + "\n")

user_message = "Show me all my notes"

print(f"User: {user_message}")
print("\nAgent:")

response = ask_agent(user_message)
print(response)


TEST 3: Listing Notes

User: Show me all my notes

Agent:
response_content Here are all your notes:

1.  **Doctor Appointment** (ID: 770c8daf-4868-4fda-8ea4-1384d9418cb7)
    Tags: personal, health
    Updated: 2025-10-25

2.  **Python Best Practices** (ID: 22101dc8-ae40-4545-9cd5-7ce3cd6bf705)
    Tags: programming, python
    Updated: 2025-10-25

3.  **Shopping List** (ID: 7d7d0e81-ab09-4b54-900c-fe1e5949cac7)
    Tags: personal, shopping
    Updated: 2025-10-25

4.  **Q4 Planning Meeting** (ID: 5acfaa64-365e-4b08-a924-5bd91dab9c48)
    Tags: work, meeting, q4
    Updated: 2025-10-25
Here are all your notes:

1.  **Doctor Appointment** (ID: 770c8daf-4868-4fda-8ea4-1384d9418cb7)
    Tags: personal, health
    Updated: 2025-10-25

2.  **Python Best Practices** (ID: 22101dc8-ae40-4545-9cd5-7ce3cd6bf705)
    Tags: programming, python
    Updated: 2025-10-25

3.  **Shopping List** (ID: 7d7d0e81-ab09-4b54-900c-fe1e5949cac7)
    Tags: personal, shopping
    Updated: 2025-10-25

4.  **Q4 Pl

## Test 4: Semantic Search

In [10]:
print("\n" + "="*80)
print("TEST 4: Semantic Search")
print("="*80 + "\n")

search_queries = [
    "Find notes about business planning",
    "What notes do I have about coding?",
    "Show me anything related to health appointments"
]

for query in search_queries:
    print(f"\nUser: {query}")
    print("\nAgent:")
    response = ask_agent(query)
    print(response)
    print("\n" + "-"*80)


TEST 4: Semantic Search


User: Find notes about business planning

Agent:
response_content I found a note that might be relevant:

**Q4 Planning Meeting**
Preview: Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on imp...
Tags: work, meeting, q4

Would you like me to show you the full content of this note or search for anything else?
I found a note that might be relevant:

**Q4 Planning Meeting**
Preview: Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on imp...
Tags: work, meeting, q4

Would you like me to show you the full content of this note or search for anything else?

--------------------------------------------------------------------------------

User: What notes do I have about coding?

Agent:
response_content I found one note about coding:

**Python Best Practices**
Preview: Always use type hints, follow PEP 8, write docstrings, use virtual environments...
Tags: programming, python


## Test 5: Keyword Search

In [11]:
print("\n" + "="*80)
print("TEST 5: Keyword Search")
print("="*80 + "\n")

user_message = "Search for notes containing the word 'customer'"

print(f"User: {user_message}")
print("\nAgent:")

response = ask_agent(user_message)
print(response)


TEST 5: Keyword Search

User: Search for notes containing the word 'customer'

Agent:
response_content I found 2 notes containing the word 'customer':

1.  **Q4 Planning Meeting**
    *Preview: Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on imp...*
    *Tags: work, meeting, q4*

2.  **Shopping List**
    *Preview: Shopping List - milk, eggs, bread, coffee, bananas...*
    *Tags: personal, shopping*
I found 2 notes containing the word 'customer':

1.  **Q4 Planning Meeting**
    *Preview: Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on imp...*
    *Tags: work, meeting, q4*

2.  **Shopping List**
    *Preview: Shopping List - milk, eggs, bread, coffee, bananas...*
    *Tags: personal, shopping*


## Test 6: Tag-based Search

In [12]:
print("\n" + "="*80)
print("TEST 6: Tag-based Search")
print("="*80 + "\n")

user_message = "Show me all notes tagged with 'personal'"

print(f"User: {user_message}")
print("\nAgent:")

response = ask_agent(user_message)
print(response)

Validation failed: Error: 'query' is required for semantic search. Please provide a search query.
2025/10/25 19:52:43 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=2000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.
2025/10/25 19:52:43 WARNING dspy.clients.lm: LM response was truncated due to exceeding max_tokens=2000. You can inspect the latest LM interactions with `dspy.inspect_history()`. To avoid truncation, consider passing a larger max_tokens when setting up dspy.LM. You may also consider increasing the temperature (currently 0.7)  if the reason for truncation is repetition.



TEST 6: Tag-based Search

User: Show me all notes tagged with 'personal'

Agent:
response_content Here are the notes tagged with 'personal':

*   **Doctor Appointment** (ID: 770c8daf-4868-4fda-8ea4-1384d941cb7)
    *   Preview: Doctor appointment on Friday at 2pm - bring insurance card...
*   **Shopping List** (ID: 7d7d0e81-ab09-4b54-900c-fe1e5949cac7)
    *   Preview: Shopping List - milk, eggs, bread, coffee, bananas...
Here are the notes tagged with 'personal':

*   **Doctor Appointment** (ID: 770c8daf-4868-4fda-8ea4-1384d941cb7)
    *   Preview: Doctor appointment on Friday at 2pm - bring insurance card...
*   **Shopping List** (ID: 7d7d0e81-ab09-4b54-900c-fe1e5949cac7)
    *   Preview: Shopping List - milk, eggs, bread, coffee, bananas...


## Test 7: Updating Notes

In [13]:
print("\n" + "="*80)
print("TEST 7: Updating Notes")
print("="*80 + "\n")

# First, get a note ID from the list
notes = await repository.list_by_user('test-user-123', limit=1)
if notes:
    note_id = notes[0].id
    print(f"Found note to update: {notes[0].title} (ID: {note_id})")
    
    user_message = f"Update the note with ID {note_id} - add to the content: 'Also need to review budget allocations'"
    
    print(f"\nUser: {user_message}")
    print("\nAgent:")
    
    response = ask_agent(user_message)
    print(response)
else:
    print("No notes available to update")


TEST 7: Updating Notes

Found note to update: Doctor Appointment (ID: 770c8daf-4868-4fda-8ea4-1384d9418cb7)

User: Update the note with ID 770c8daf-4868-4fda-8ea4-1384d9418cb7 - add to the content: 'Also need to review budget allocations'

Agent:
response_content I've updated the note titled 'Doctor Appointment' (ID: 770c8daf-4868-4fda-8ea4-1384d9418cb7) by adding 'Also need to review budget allocations' to its content.
I've updated the note titled 'Doctor Appointment' (ID: 770c8daf-4868-4fda-8ea4-1384d9418cb7) by adding 'Also need to review budget allocations' to its content.


## Test 8: Error Handling - Invalid Input

In [14]:
print("\n" + "="*80)
print("TEST 8: Error Handling - Invalid Input")
print("="*80 + "\n")

error_cases = [
    "Create a note with empty title: '' and content 'test'",
    "Delete note with ID 123 (should be string, not number)",
    "Get note with ID 'nonexistent-note-id'"
]

for error_case in error_cases:
    print(f"\nUser: {error_case}")
    print("\nAgent:")
    response = ask_agent(error_case)
    print(response)
    print("\n" + "-"*80)

Validation failed: Error: 'title' must be a non-empty string.
Note 123 not found for deletion



TEST 8: Error Handling - Invalid Input


User: Create a note with empty title: '' and content 'test'

Agent:
response_content I'm sorry, I couldn't create the note with an empty title. The note system requires a non-empty title for all notes.
I'm sorry, I couldn't create the note with an empty title. The note system requires a non-empty title for all notes.

--------------------------------------------------------------------------------

User: Delete note with ID 123 (should be string, not number)

Agent:
response_content I'm sorry, I couldn't find a note with the ID "123" to delete. It seems it doesn't exist.
I'm sorry, I couldn't find a note with the ID "123" to delete. It seems it doesn't exist.

--------------------------------------------------------------------------------

User: Get note with ID 'nonexistent-note-id'

Agent:
response_content Note not found with ID 'nonexistent-note-id'.
Note not found with ID 'nonexistent-note-id'.

--------------------------------------------

## Test 9: Multi-turn Conversation

In [15]:
print("\n" + "="*80)
print("TEST 9: Multi-turn Conversation")
print("="*80 + "\n")

conversation_history = []

def chat(message):
    print(f"\n👤 User: {message}")
    response = ask_agent(message, conversation_history)
    print(f"\n🤖 Agent: {response}")
    
    # Update history
    conversation_history.append({"role": "user", "content": message})
    conversation_history.append({"role": "assistant", "content": response})
    
    return response

# Multi-turn conversation
chat("Create a note about a new project idea: 'Build an AI-powered task manager with natural language interface'")
chat("Add tags 'project', 'ai', 'idea' to that note")
chat("What notes do I have about AI?")
chat("Show me the details of the first one")


TEST 9: Multi-turn Conversation


👤 User: Create a note about a new project idea: 'Build an AI-powered task manager with natural language interface'
response_content I've created a note titled 'Project Idea' with the content: 'Build an AI-powered task manager with natural language interface'.

🤖 Agent: I've created a note titled 'Project Idea' with the content: 'Build an AI-powered task manager with natural language interface'.

👤 User: Add tags 'project', 'ai', 'idea' to that note


Validation failed: Error: Invalid search_type 'title'. Valid types are: semantic, keyword, tag, date.


response_content I've added the tags 'project', 'ai', and 'idea' to the 'Project Idea' note.

🤖 Agent: I've added the tags 'project', 'ai', and 'idea' to the 'Project Idea' note.

👤 User: What notes do I have about AI?


Validation failed: Error: 'query' is required for semantic search. Please provide a search query.


response_content I found one note explicitly tagged with 'ai':

**Project Idea**
*Content: Build an AI-powered task manager with natural language interface*
*Tags: project, ai, idea*

Would you like to see the full content of this note or perhaps search for notes with different criteria?

🤖 Agent: I found one note explicitly tagged with 'ai':

**Project Idea**
*Content: Build an AI-powered task manager with natural language interface*
*Tags: project, ai, idea*

Would you like to see the full content of this note or perhaps search for notes with different criteria?

👤 User: Show me the details of the first one


Validation failed: Error: Invalid search_type 'title'. Valid types are: semantic, keyword, tag, date.


response_content Here are the full details for the 'Project Idea' note:

**Title:** Project Idea
**Content:** Build an AI-powered task manager with natural language interface
**Tags:** project, ai, idea
**Created At:** 2025-10-25T11:53:03.469029+00:00
**Updated At:** 2025-10-25T11:53:17.125629+00:00

🤖 Agent: Here are the full details for the 'Project Idea' note:

**Title:** Project Idea
**Content:** Build an AI-powered task manager with natural language interface
**Tags:** project, ai, idea
**Created At:** 2025-10-25T11:53:03.469029+00:00
**Updated At:** 2025-10-25T11:53:17.125629+00:00


"Here are the full details for the 'Project Idea' note:\n\n**Title:** Project Idea\n**Content:** Build an AI-powered task manager with natural language interface\n**Tags:** project, ai, idea\n**Created At:** 2025-10-25T11:53:03.469029+00:00\n**Updated At:** 2025-10-25T11:53:17.125629+00:00"

## Test 10: Complex Query

In [16]:
print("\n" + "="*80)
print("TEST 10: Complex Query")
print("="*80 + "\n")

user_message = """I need help organizing my notes. Can you:
1. Tell me how many notes I have
2. Show me notes related to work
3. Give me a summary of what topics I'm tracking
"""

print(f"User: {user_message}")
print("\nAgent:")

response = ask_agent(user_message)
print(response)


TEST 10: Complex Query

User: I need help organizing my notes. Can you:
1. Tell me how many notes I have
2. Show me notes related to work
3. Give me a summary of what topics I'm tracking


Agent:


Validation failed: Error: 'query' is required for semantic search. Please provide a search query.
Validation failed: Error: Invalid search_type 'tags'. Valid types are: semantic, keyword, tag, date.


response_content Certainly! Here's an overview of your notes:

1.  **Total Notes:** You currently have **5** notes.

2.  **Notes related to work:**
    *   **Q4 Planning Meeting** (Tags: work, meeting, q4)
        *   Preview: "Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on imp..."

3.  **Summary of topics you're tracking:**
    Based on the tags across all your notes, you are tracking topics such as: `project`, `ai`, `idea`, `personal`, `health`, `programming`, `python`, `shopping`, `work`, `meeting`, and `q4`.

Let me know if you'd like to dive deeper into any of these!
Certainly! Here's an overview of your notes:

1.  **Total Notes:** You currently have **5** notes.

2.  **Notes related to work:**
    *   **Q4 Planning Meeting** (Tags: work, meeting, q4)
        *   Preview: "Discussed product roadmap, customer retention strategies, and team expansion plans. Key focus on imp..."

3.  **Summary of topics you're tracking:**
    Based o

## Repository Statistics

In [17]:
print("\n" + "="*80)
print("REPOSITORY STATISTICS")
print("="*80 + "\n")

total_notes = await repository.count_by_user('test-user-123')
all_notes = await repository.list_by_user('test-user-123', limit=100)

print(f"Total notes: {total_notes}")
print(f"\nNotes breakdown:")

# Collect all tags
all_tags = set()
for note in all_notes:
    all_tags.update(note.tags)

print(f"Unique tags: {', '.join(sorted(all_tags)) if all_tags else 'none'}")

# Show all notes
print(f"\nAll notes:")
for i, note in enumerate(all_notes, 1):
    print(f"\n{i}. {note.title}")
    print(f"   ID: {note.id}")
    print(f"   Content: {note.content[:100]}..." if len(note.content) > 100 else f"   Content: {note.content}")
    print(f"   Tags: {', '.join(note.tags) if note.tags else 'none'}")
    print(f"   Created: {note.created_at.isoformat()}")
    print(f"   Has embedding: {'Yes' if note.embedding else 'No'}")


REPOSITORY STATISTICS

Total notes: 5

Notes breakdown:
Unique tags: ai, health, idea, meeting, personal, programming, project, python, q4, shopping, work

All notes:

1. Project Idea
   ID: 6f3142e9-e287-4689-ba94-1d5ba4671f58
   Content: Build an AI-powered task manager with natural language interface
   Tags: project, ai, idea
   Created: 2025-10-25T11:53:03.469029+00:00
   Has embedding: Yes

2. Doctor Appointment
   ID: 770c8daf-4868-4fda-8ea4-1384d9418cb7
   Content: Also need to review budget allocations
   Tags: personal, health
   Created: 2025-10-25T11:52:11.500490+00:00
   Has embedding: Yes

3. Python Best Practices
   ID: 22101dc8-ae40-4545-9cd5-7ce3cd6bf705
   Content: Always use type hints, follow PEP 8, write docstrings, use virtual environments
   Tags: programming, python
   Created: 2025-10-25T11:52:07.151773+00:00
   Has embedding: Yes

4. Shopping List
   ID: 7d7d0e81-ab09-4b54-900c-fe1e5949cac7
   Content: Shopping List - milk, eggs, bread, coffee, bananas
   Tag

## Summary

This notebook demonstrates:

✅ **Note Creation**: Agent can create notes from natural language
✅ **Semantic Search**: Vector-based search finds notes by meaning
✅ **Keyword Search**: Text matching in title/content
✅ **Tag Search**: Filter by tags
✅ **List & Get**: View all notes or specific ones
✅ **Update**: Modify existing notes
✅ **Error Handling**: Plain English error messages
✅ **Multi-turn**: Conversation context maintained
✅ **Complex Queries**: Handle multi-step requests

The agent seamlessly uses the 3-layer note_tools architecture to provide a natural note-taking experience!